In [1]:
import sys
import logging
import pandas as pd
import duckdb
from faker import Faker

# ==========================================
# 1. SETUP PRODUCTION LOGGING
# ==========================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] (%(filename)s:%(lineno)d) - %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger("ELT_Pipeline")

# ==========================================
# 2. INITIALIZE WAREHOUSE CONNECTION
# ==========================================
def initialize_warehouse(db_path: str = "analytics_dw.duckdb") -> duckdb.DuckDBPyConnection:
    """
    Establishes connection to local DuckDB analytical engine.
    Creates persistent database file if it does not exist.
    """
    try:
        logger.info(f"Connecting to DuckDB Data Warehouse at target: '{db_path}'...")
        conn = duckdb.connect(database=db_path, read_only=False)
        
        # Verify connection with engine metadata query
        version = conn.execute("SELECT version();").fetchone()[0]
        logger.info(f"Successfully connected to DuckDB Engine version: {version}")
        
        return conn
    except Exception as e:
        logger.error(f"Fatal error initializing data warehouse connection: {e}", exc_info=True)
        raise

# Execute connection test
if __name__ == "__main__":
    try:
        db_conn = initialize_warehouse()
        print("\n--- PHASE 1 ENVIRONMENT VERIFIED ---")
    except Exception:
        print("\n--- PHASE 1 INITIALIZATION FAILED ---")

2026-08-01 04:42:07,119 [INFO] (3036449790.py:28) - Connecting to DuckDB Data Warehouse at target: 'analytics_dw.duckdb'...
2026-08-01 04:42:07,133 [INFO] (3036449790.py:33) - Successfully connected to DuckDB Engine version: v1.5.4

--- PHASE 1 ENVIRONMENT VERIFIED ---


# PHASE - 2

# 1. 🏗️ Architecture Context
We are currently at the absolute beginning of the pipeline: the Extract phase.
Our goal is to pull real-world dimension data from a mock e-commerce API and computationally simulate 1,000,000+ transactional records for Orders and Customer Experience (CX) Tickets.

In [5]:
# [ REST API ] -------------- (requests) ----> [ Product Catalog DataFrame ]
#                                                          │
#  Synthetic Engine ] ------ (Faker/random)-> [ 1M+ Orders DataFrame ]
#                                                           │
#[ Synthetic Engine ] ------ (Faker/random)-> [ 1M+ CX Tickets DataFrame ]

In [2]:
import requests
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

# Initialize Faker
fake = Faker()
Faker.seed(42)
random.seed(42)

# Pipeline Configuration
NUM_RECORDS = 1_000_000

# ==========================================
# 1. EXTRACT: PRODUCT DIMENSION FROM API
# ==========================================
def extract_products() -> pd.DataFrame:
    """Fetches real e-commerce product data from a mock REST API."""
    logger.info("Extracting product catalog from FakeStoreAPI...")
    url = "https://fakestoreapi.com/products"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        products_data = response.json()
        
        df_products = pd.DataFrame(products_data)[['id', 'title', 'category', 'price']]
        df_products.rename(columns={'id': 'product_id'}, inplace=True)
        logger.info(f"Successfully extracted {len(df_products)} products.")
        return df_products
    
    except requests.exceptions.RequestException as e:
        logger.error(f"API Extraction failed: {e}")
        raise

# ==========================================
# 2. GENERATE: 1 MILLION ORDERS (FACT TABLE)
# ==========================================
def generate_orders(num_rows: int, product_ids: list) -> pd.DataFrame:
    """Generates a massive synthetic orders table using optimized batching."""
    logger.info(f"Generating {num_rows:,} synthetic orders. This may take ~10 seconds...")
    
    # Pre-generate dates for speed rather than calling Faker 1M times
    end_date = datetime.now()
    start_date = end_date - timedelta(days=730) # 2 years of data
    
    orders = {
        "order_id": range(100000, 100000 + num_rows),
        "customer_id": [random.randint(1, 50000) for _ in range(num_rows)],
        "product_id": [random.choice(product_ids) for _ in range(num_rows)],
        # Messy data: Some orders have negative amounts (refund anomalies)
        "amount": [round(random.uniform(-10.0, 500.0), 2) for _ in range(num_rows)],
        "order_date": [start_date + timedelta(seconds=random.randint(0, int(63072000))) for _ in range(num_rows)]
    }
    
    df_orders = pd.DataFrame(orders)
    logger.info("Orders generation complete.")
    return df_orders

# ==========================================
# 3. GENERATE: 1 MILLION CX TICKETS (FACT TABLE)
# ==========================================
def generate_tickets(num_rows: int, order_ids: list) -> pd.DataFrame:
    """Generates CX Zendesk-style tickets linked to our orders."""
    logger.info(f"Generating {num_rows:,} synthetic CX tickets...")
    
    issue_types = ['Late Delivery', 'Damaged Item', 'Billing Error', 'General Inquiry', 'Return Request']
    
    # We simulate a scenario where not all tickets have CSAT scores (messy data)
    tickets = {
        "ticket_id": [f"TCK-{i}" for i in range(1, num_rows + 1)],
        "order_id": [random.choice(order_ids) for _ in range(num_rows)],
        "issue_type": [random.choice(issue_types) for _ in range(num_rows)],
        "csat_score": [random.choice([1, 2, 3, 4, 5, None]) for _ in range(num_rows)],
        "is_resolved": [random.choice([True, False]) for _ in range(num_rows)]
    }
    
    df_tickets = pd.DataFrame(tickets)
    logger.info("CX Tickets generation complete.")
    return df_tickets

# Execute Pipeline Extract/Generate
if __name__ == "__main__":
    try:
        df_products = extract_products()
        valid_product_ids = df_products['product_id'].tolist()
        
        df_orders = generate_orders(NUM_RECORDS, valid_product_ids)
        valid_order_ids = df_orders['order_id'].tolist()
        
        df_tickets = generate_tickets(NUM_RECORDS, valid_order_ids)
        
        logger.info("--- PHASE 2 EXTRACTION & GENERATION SUCCESSFUL ---")
    except Exception as e:
        logger.error("Phase 2 Failed.", exc_info=True)

2026-08-01 04:52:44,419 [INFO] (721291171.py:20) - Extracting product catalog from FakeStoreAPI...
2026-08-01 04:52:45,269 [INFO] (721291171.py:30) - Successfully extracted 20 products.
2026-08-01 04:52:45,271 [INFO] (721291171.py:42) - Generating 1,000,000 synthetic orders. This may take ~10 seconds...
2026-08-01 04:52:46,859 [INFO] (721291171.py:58) - Orders generation complete.
2026-08-01 04:52:46,894 [INFO] (721291171.py:66) - Generating 1,000,000 synthetic CX tickets...
2026-08-01 04:52:48,067 [INFO] (721291171.py:80) - CX Tickets generation complete.
2026-08-01 04:52:48,079 [INFO] (721291171.py:94) - --- PHASE 2 EXTRACTION & GENERATION SUCCESSFUL ---


In [3]:

print(f'Products shape: {df_products.shape}')
print(f'Orders shape: {df_orders.shape}')
print(f'Tickets shape: {df_tickets.shape}')

Products shape: (20, 4)
Products shape: (1000000, 5)
Products shape: (1000000, 5)


# Phase 3: The Load (Bronze/Raw Layer).

# 1. 🏗️ Architecture Context
In modern data engineering, we use a Medallion Architecture (Bronze -> Silver -> Gold). Right now, we are building the Bronze Layer. This means loading the data exactly as it is—messy, raw, and untransformed—into the warehouse.

In [4]:
# [ Python Memory (Pandas) ]
#            │
#            ▼ (DuckDB Zero-Copy Ingestion)
#[ Data Warehouse (Bronze Layer) ]
#      ├── raw_products (20 rows)
#      ├── raw_orders   (1,000,000 rows)
#      └── raw_tickets  (1,000,000 rows)

# 2. 💼 The Production Standard (Why we do it this way)
Why ELT instead of ETL? Historically, engineers used ETL (Extract, Transform, Load), meaning they used Python or Java to clean the data before loading it into the database. Today, compute is cheap and databases are incredibly fast. We use ELT (Extract, Load, Transform). We dump the raw data into the warehouse first. Why? Because if our transformation logic breaks, we don't have to re-ping the API or re-run the extraction. The raw data is safely stored on disk, ready to be re-processed.

Why DuckDB is Magic: Normally, you would have to write sluggish INSERT INTO SQL loops or bulk copy commands to load a database. DuckDB has a feature called zero-copy integration. It can read Pandas DataFrames directly as if they were SQL tables. We can create our warehouse tables in milliseconds.

In [6]:
# ==========================================
# 4. LOAD: BRONZE LAYER (RAW DATA)
# ==========================================
def load_bronze_layer(conn: duckdb.DuckDBPyConnection):
    """
    Ingests Pandas DataFrames into DuckDB as persistent tables.
    Uses DuckDB's native DataFrame scanning capability.
    """
    logger.info("Initiating Bronze layer ingestion...")
    
    try:
        # Note: DuckDB automatically sees the DataFrames 'df_products', 'df_orders', 
        # and 'df_tickets' currently active in your Python environment.
        
        logger.info("Writing raw_products to warehouse...")
        conn.execute("""
            CREATE OR REPLACE TABLE raw_products AS 
            SELECT * FROM df_products;
        """)
        
        logger.info("Writing raw_orders (1M+ rows) to warehouse...")
        conn.execute("""
            CREATE OR REPLACE TABLE raw_orders AS 
            SELECT * FROM df_orders;
        """)
        
        logger.info("Writing raw_tickets (1M+ rows) to warehouse...")
        conn.execute("""
            CREATE OR REPLACE TABLE raw_tickets AS 
            SELECT * FROM df_tickets;
        """)
        
        logger.info("Bronze layer successfully loaded to disk.")
        
    except Exception as e:
        logger.error(f"Fatal error during Bronze layer load: {e}", exc_info=True)
        raise

# Execute Bronze Load
if __name__ == "__main__":
    try:
        # We reuse the db_conn initialized in Phase 1
        load_bronze_layer(db_conn)
        
        logger.info("--- PHASE 3 BRONZE LOAD SUCCESSFUL ---")
    except Exception as e:
        logger.error("Phase 3 Failed.", exc_info=True)

2026-08-01 05:04:58,300 [INFO] (3598948207.py:9) - Initiating Bronze layer ingestion...
2026-08-01 05:04:58,302 [INFO] (3598948207.py:15) - Writing raw_products to warehouse...
2026-08-01 05:04:58,490 [INFO] (3598948207.py:21) - Writing raw_orders (1M+ rows) to warehouse...
2026-08-01 05:04:58,656 [INFO] (3598948207.py:27) - Writing raw_tickets (1M+ rows) to warehouse...
2026-08-01 05:04:59,106 [INFO] (3598948207.py:33) - Bronze layer successfully loaded to disk.
2026-08-01 05:04:59,106 [INFO] (3598948207.py:45) - --- PHASE 3 BRONZE LOAD SUCCESSFUL ---


In [9]:
# Query the warehouse metadata and row counts
validate_query = """
    SELECT 
        table_name,
        estimated_size as row_count
    FROM duckdb_tables()
"""
print(db_conn.execute(validate_query).fetchdf())

     table_name  row_count
0    raw_orders    1000000
1  raw_products         20
2   raw_tickets    1000000


# Phase 4: The Transform (Silver/Gold Layer via SQL).

# 1. 🏗️ Architecture Context
We are now implementing the Medallion Architecture. We will read the messy Bronze tables, clean and enrich them in a Silver layer, and finally join them into a business-ready Gold table that our executives and downstream dashboards will consume.

In [10]:
#[ Bronze (Raw) ] 
#       │
#       ▼ (SQL: CTEs, Window Functions, Case Statements)
#[ Silver (Cleaned) ]
#       │
#       ▼ (SQL: Heavy Joins)
#[ Gold_Customer_Experience (Business Ready Table) ]

# 2. 💼 The Production Standard (Why we do it this way)
Why use SQL for this instead of Pandas? If you tried to join a 1-million-row order table with a 1-million-row ticket table in Pandas, your laptop's memory would likely spike and crash (OOM). DuckDB (like Snowflake or BigQuery) uses disk-spilling and vectorized execution. It handles joins of this size in a fraction of a second. We use Python as the orchestrator, but we push the heavy compute down to the SQL engine.

Why Common Table Expressions (CTEs)? Junior analysts write massive, unreadable nested subqueries. Senior engineers write CTEs (using the WITH clause) to break complex logic into readable, testable, and modular steps.

In [11]:
# ==========================================
# 5. TRANSFORM: SILVER & GOLD LAYERS
# ==========================================
def run_transformations(conn: duckdb.DuckDBPyConnection):
    """
    Executes native SQL transformations in the warehouse.
    Builds the Gold business-ready table utilizing CTEs and Window Functions.
    """
    logger.info("Initiating SQL Transformations (Bronze -> Gold)...")
    
    transform_query = """
        -- 1. Create our final business-ready Gold table
        CREATE OR REPLACE TABLE gold_customer_experience AS
        
        -- 2. SILVER ORDERS: Clean anomalies and calculate window metrics
        WITH cleaned_orders AS (
            SELECT 
                order_id,
                customer_id,
                product_id,
                -- Clean messy data: Convert negative refund amounts to 0
                CASE WHEN amount < 0 THEN 0 ELSE amount END AS clean_amount,
                order_date,
                -- Window Function 1: Rank order sequence per customer
                ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY order_date) as customer_order_seq,
                -- Window Function 2: Total lifetime spend per customer
                SUM(CASE WHEN amount < 0 THEN 0 ELSE amount END) OVER(PARTITION BY customer_id) as lifetime_spend
            FROM raw_orders
        ),
        
        -- 3. SILVER TICKETS: Clean nulls and define SLAs
        cleaned_tickets AS (
            SELECT 
                ticket_id,
                order_id,
                issue_type,
                -- Handle missing data: Default null CSATs to a neutral 3
                COALESCE(csat_score, 3) AS imputed_csat,
                is_resolved,
                -- Simulate SLA logic
                CASE WHEN is_resolved = FALSE THEN 'SLA Breached' ELSE 'Within SLA' END AS sla_status
            FROM raw_tickets
        )
        
        -- 4. GOLD JOIN: Combine domains and apply business routing logic
        SELECT 
            t.ticket_id,
            t.issue_type,
            t.imputed_csat,
            t.sla_status,
            o.customer_id,
            o.clean_amount AS order_value,
            o.lifetime_spend,
            o.customer_order_seq,
            p.title AS product_name,
            p.category AS product_category,
            
            -- Complex Business Logic: Flag urgent tickets for VIPs
            CASE 
                WHEN o.lifetime_spend > 2500 AND t.sla_status = 'SLA Breached' 
                THEN 'URGENT - High Value VIP'
                ELSE 'Standard' 
            END AS routing_priority
            
        FROM cleaned_tickets t
        JOIN cleaned_orders o ON t.order_id = o.order_id
        JOIN raw_products p ON o.product_id = p.product_id;
    """
    
    try:
        logger.info("Executing massive SQL Transform query. Engine processing...")
        conn.execute(transform_query)
        logger.info("Transformation complete. Table 'gold_customer_experience' persisted to disk.")
    except Exception as e:
        logger.error(f"Transformation Pipeline Failed: {e}", exc_info=True)
        raise

# Execute the Transformation
if __name__ == "__main__":
    try:
        run_transformations(db_conn)
        logger.info("--- PHASE 4 TRANSFORMATION SUCCESSFUL ---")
    except Exception as e:
        logger.error("Phase 4 Failed.", exc_info=True)

2026-08-01 05:12:19,983 [INFO] (2406300209.py:9) - Initiating SQL Transformations (Bronze -> Gold)...
2026-08-01 05:12:19,984 [INFO] (2406300209.py:71) - Executing massive SQL Transform query. Engine processing...
2026-08-01 05:12:21,092 [INFO] (2406300209.py:73) - Transformation complete. Table 'gold_customer_experience' persisted to disk.
2026-08-01 05:12:21,095 [INFO] (2406300209.py:82) - --- PHASE 4 TRANSFORMATION SUCCESSFUL ---


In [12]:
# Query the new Gold table
validation_query_2 = """
    SELECT 
        routing_priority, 
        COUNT(*) as ticket_count 
    FROM gold_customer_experience 
    GROUP BY routing_priority
    ORDER BY ticket_count DESC;
"""
print(db_conn.execute(validation_query_2).fetchdf())

          routing_priority  ticket_count
0                 Standard        506498
1  URGENT - High Value VIP        493502


# Phase 5: The Analytics (Serving Layer). It is time to deliver the ROI of this pipeline to our stakeholders.

# 1. 🏗️ Architecture Context
We have our beautifully modeled gold_customer_experience table sitting on disk in DuckDB. Now, we need to serve these insights to the business (e.g., the VP of Customer Experience). We will query the aggregated data out of the warehouse and back into Python to calculate final KPIs like Net Promoter Score (NPS) and SLA Breach Rates.

In [13]:
#[ Data Warehouse (Gold Layer) ]
#               │
#               ▼ (SQL Aggregation: GROUP BY)
#[ Python Engine (Pandas) ]
#               │
#               ▼ (Python: Metric Math & Formatting)
#[ Executive KPI Dashboard (Stdout / BI Tool) ]

# 2. 💼 The Production Standard (Why we do it this way)
Why bring data back into Python now? SQL is unmatched for joining massive tables and grouping data. But once the data is aggregated down to a few dozen rows, Python shines. We use Pandas or Python visualization libraries (like Plotly or Altair) to calculate final derived percentages, format the output, or push the clean summary directly to a BI tool API (like Tableau, Looker, or Streamlit).

The Hybrid Sweet Spot: Push the heavy lifting (millions of rows) to the SQL engine. Pull the lightweight results (dozens of rows) into Python for the final polish.

In [14]:
# ==========================================
# 6. ANALYTICS: THE SERVING LAYER
# ==========================================
def generate_executive_dashboard(conn: duckdb.DuckDBPyConnection) -> pd.DataFrame:
    """
    Queries the Gold layer for aggregated metrics and uses Pandas 
    to calculate final KPI percentages for stakeholders.
    """
    logger.info("Extracting aggregate data for Executive Dashboard...")
    
    # 1. THE SERVING QUERY (SQL for Heavy Aggregation)
    analytics_query = """
        SELECT 
            product_category,
            COUNT(ticket_id) AS total_tickets,
            SUM(CASE WHEN sla_status = 'SLA Breached' THEN 1 ELSE 0 END) AS breached_tickets,
            -- Define Promoters (CSAT 4-5) and Detractors (CSAT 1-3)
            SUM(CASE WHEN imputed_csat >= 4 THEN 1 ELSE 0 END) AS promoters,
            SUM(CASE WHEN imputed_csat <= 3 THEN 1 ELSE 0 END) AS detractors
        FROM gold_customer_experience
        GROUP BY product_category;
    """
    
    try:
        # Fetch the aggregated data back into a Pandas DataFrame
        df_kpis = conn.execute(analytics_query).fetchdf()
        
        # 2. FINAL PYTHON TRANSFORMATIONS (Lightweight Math & Formatting)
        # Calculate SLA Breach Rate %
        df_kpis['sla_breach_rate_%'] = (
            (df_kpis['breached_tickets'] / df_kpis['total_tickets']) * 100
        ).round(1)
        
        # Calculate approximate NPS Score: % Promoters - % Detractors
        df_kpis['nps_score'] = (
            ((df_kpis['promoters'] - df_kpis['detractors']) / df_kpis['total_tickets']) * 100
        ).round(1)
        
        # Sort for the stakeholder view (highest ticket volume first)
        df_dashboard = df_kpis.sort_values(by='total_tickets', ascending=False)
        
        logger.info("Dashboard generation complete.")
        return df_dashboard
        
    except Exception as e:
        logger.error(f"Failed to generate Executive Dashboard: {e}", exc_info=True)
        raise

# Execute the Serving Layer
if __name__ == "__main__":
    try:
        df_exec_dashboard = generate_executive_dashboard(db_conn)
        
        print("\n" + "="*50)
        print(" 📊 EXECUTIVE CX DASHBOARD (Serving Layer)")
        print("="*50)
        # Display the final polished metrics
        print(df_exec_dashboard[['product_category', 'total_tickets', 'sla_breach_rate_%', 'nps_score']].to_string(index=False))
        print("="*50)
        
        # Close the connection like a professional
        db_conn.close()
        logger.info("Pipeline execution finished. Warehouse connection closed.")
        
    except Exception as e:
        logger.error("Phase 5 Failed.", exc_info=True)

2026-08-01 05:15:25,027 [INFO] (3241691279.py:9) - Extracting aggregate data for Executive Dashboard...
2026-08-01 05:15:25,292 [INFO] (3241691279.py:42) - Dashboard generation complete.

 📊 EXECUTIVE CX DASHBOARD (Serving Layer)
product_category  total_tickets  sla_breach_rate_%  nps_score
     electronics         300595               50.0      -33.3
women's clothing         299184               50.1      -33.4
  men's clothing         200625               49.9      -33.0
        jewelery         199596               49.8      -33.0
2026-08-01 05:15:25,322 [INFO] (3241691279.py:63) - Pipeline execution finished. Warehouse connection closed.


In [ ]:
#[ FakeStore API ]                  [ Faker Synthetic Engine ]
#        │                                      │
#        ▼ (Python requests)                    ▼ (Python lists/generators)
#[ 20 Products df ]                 [ 1M Orders df / 1M Tickets df ]
#        │                                      │
#        └───────────────────┬──────────────────┘
#                            ▼ (DuckDB Zero-Copy Ingestion)
#                  [ Bronze Layer (Raw Disk) ]
#                            │
#                            ▼ (SQL CTEs, Cleaning, Null Handling)
#                  [ Silver Layer (Cleaned) ]
#                            │
#                            ▼ (SQL Joins, Window Functions)
#                  [ Gold Layer (Business Logic) ]
#                            │
#                            ▼ (SQL Heavy Aggregation -> Pandas Math)
#               [ Final Executive CX Dashboard ]

# 2. 💼 The Production Standard (What I achieved today)
Decoupled Extraction: You used Python to talk to the outside world, pulling API data and generating scale, but kept it out of the heavy transformation layer.

Disk-Backed Processing: Instead of crashing Pandas with a 2-million row join, you loaded everything to disk in a columnar OLAP engine (DuckDB), simulating how a company uses Snowflake or BigQuery.

Modular SQL inside Python: You used WITH statements (CTEs) to separate cleaning from joining, making your data models readable, testable, and maintainable.

Graceful Shutdown: You closed your database connection like a professional, preventing locked files and memory leaks.